In [ ]:
import pandas as pd

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# Preparing the data for training (split into train, test and validation sets)

In [8]:
OUTPUT_DIR = Path("../Data/processed/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
data_path = Path("../Data/processed/discharge_cleaned.csv")

data = pd.read_csv(data_path)

In [9]:
# Split data into train, test and validation sets using group shuffle to prevent same patients appearing in all three splits
data = data[data["cleaned_text"].str.strip() != ""]

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 42)

# 80% train, 20% temporary for text and validation tests
train_idx , temp_idx = next(gss.split(data, groups = data["subject_id"]))

train_df = data.iloc[train_idx].copy()
temp_df = data.iloc[temp_idx].copy()

# Split temporary set into 10% validation and 10% test
gss2 = GroupShuffleSplit(n_splits = 1, test_size = 0.5, random_state = 42)

valid_idx, test_idx = next(gss2.split(temp_df, groups = temp_df["subject_id"]))

valid_df = temp_df.iloc[valid_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

In [11]:
def save_txt(data, output_path):
    with open(output_path, "w", encoding = "utf-8") as f:
        for text in data["cleaned_text"]:
            if pd.notna(text) and isinstance(text, str) and len(text.strip()) != 0:
                f.write(text.strip())
                f.write("\n<|endoftext|>\n")

save_txt(train_df, OUTPUT_DIR / "train.txt")
save_txt(valid_df, OUTPUT_DIR / "valid.txt")
save_txt(test_df, OUTPUT_DIR / "test.txt")

print("Saved train, valid and test txt files")

Saved train, valid and test txt files
